In [2]:

import os
import asyncio
import json
import sys
import pathlib



from dotenv import load_dotenv
load_dotenv()  # 读取 backend/.env

from loguru import logger
from agent.task_queue import (
    enqueue_task,
    start_worker,
    read_task_events,
    TASK_STREAM,
)
from agent.logger import setup_logger
from agent.task_queue import _process_one_task, _get_redis, _ensure_consumer_group
import redis.asyncio as redis

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379/0")
TASK_STREAM = "research:tasks"
EVENT_STREAM_PREFIX = "research:events"
CONSUMER_GROUP = "research-workers"
# 单个任务的事件流最大保留条数，防止无限增长
EVENT_STREAM_MAXLEN = 500
# XREAD 阻塞超时（毫秒），避免空轮询
STREAM_BLOCK_MS = 5000
setup_logger()


<loguru.logger handlers=[(id=3, level=20, sink=stderr), (id=4, level=10, sink='logs/ZhiPoAI_DR_{time:YYYY-MM-DD}.log')]>

In [6]:

# ════════════════════════════════════════════════════════════════════
# Step 3: 模拟 GET /api/research/{task_id}/stream  (SSE)
# ════════════════════════════════════════════════════════════════════
async def simulate_sse_stream(task_id: str, max_events: int = 200, last_event_id: str = "0"):
    """模拟 GET /api/research/{task_id}/stream ——
    调用 read_task_events 的内部 Generator，逐条打印事件。
    """
    print("\n" + "=" * 70)
    print(f"  [Step 3] 模拟 SSE 事件流读取")
    print("=" * 70)

    count = 0
    async for event_str in read_task_events(task_id, last_event_id=last_event_id):
        count += 1
        # 模拟 SSE:  "data: <event>\n\n"
        print(f"\n  ┌─ event[{count}] ─────────────────────────────────")
        try:
            event = json.loads(event_str)
            print(f"  │  {json.dumps(event, ensure_ascii=False, indent=2)[:1500]}")
        except json.JSONDecodeError:
            print(f"  │  {event_str[:1500]}")

        # 检测终止事件
        try:
            event = json.loads(event_str)
            if any(k in event for k in ("finalize_answer", "error", "task_paused")):
                print(f"  └─ 终止事件 → SSE 连接关闭")
                break
        except json.JSONDecodeError:
            pass

        if count >= max_events:
            print(f"  └─ 达到 max_events={max_events}，提前断开")
            break



In [4]:
task_id = '5c4e8aee-8de5-4f32-85e6-6a42ffaab7fc'

In [9]:

LAST_EVENT_ID = '1789280258419-0'
await simulate_sse_stream(task_id, last_event_id=LAST_EVENT_ID)



  [Step 3] 模拟 SSE 事件流读取

  ┌─ event[1] ─────────────────────────────────
  │  {
  "token": {
    "text": "（如",
    "node": "generate_plan"
  }
}

  ┌─ event[2] ─────────────────────────────────
  │  {
  "token": {
    "text": "2026",
    "node": "generate_plan"
  }
}

  ┌─ event[3] ─────────────────────────────────
  │  {
  "token": {
    "text": "-2030年",
    "node": "generate_plan"
  }
}

  ┌─ event[4] ─────────────────────────────────
  │  {
  "token": {
    "text": "市场规模",
    "node": "generate_plan"
  }
}

  ┌─ event[5] ─────────────────────────────────
  │  {
  "token": {
    "text": "C",
    "node": "generate_plan"
  }
}

  ┌─ event[6] ─────────────────────────────────
  │  {
  "token": {
    "text": "AGR）\n\n---\n\n##",
    "node": "generate_plan"
  }
}

  ┌─ event[7] ─────────────────────────────────
  │  {
  "token": {
    "text": " 下一步：\n请",
    "node": "generate_plan"
  }
}

  ┌─ event[8] ─────────────────────────────────
  │  {
  "token": {
    "text": "逐",
    "node": "g